# End-to-End Speech Recognition with FunASR Nano and OpenVINO

[FunASR Nano](https://huggingface.co/FunAudioLLM/Fun-ASR-Nano-2512) is an end-to-end speech recognition large model launched by Tongyi Lab. It is trained on tens of millions of hours of real speech data, supporting low-latency real-time transcription across 31 languages. It excels in vertical domains such as education and finance, accurately recognizing professional terminology and regional accents.

**Model architecture** -- FunASR Nano (~800M parameters) is a **multimodal audio-language model** with four components:

| Component | Role |
|---|---|
| **Audio Frontend (WavFrontend)** | Extracts Fbank features from raw waveform (mel-frequency filterbank) |
| **Audio Encoder** | Converts audio features into audio embeddings |
| **Text Embeddings** | Standard token embeddings for the LLM vocabulary |
| **Language Model (Qwen3-0.6B)** | Generates transcription from merged audio + text embeddings |

The pipeline works as: **Audio -> Frontend -> Encoder -> Embeddings merge with text prompt -> LLM -> Transcribed text**

In this tutorial we demonstrate how to convert, run, and optimize FunASR Nano using **OpenVINO** and discuss **OpenVINO GenAI** integration.

#### Table of contents:

- [1. Environment Setup](#1.-Environment-Setup)
- [2. Hugging Face Authentication](#2.-Hugging-Face-Authentication)
- [3. Model Download and Analysis](#3.-Model-Download-and-Analysis)
- [4. Conversion to OpenVINO IR](#4.-Conversion-to-OpenVINO-IR)
- [5. OpenVINO Runtime Inference](#5.-OpenVINO-Runtime-Inference)
- [6. Multi-Device Inference (CPU / GPU / NPU)](#6.-Multi-Device-Inference-(CPU-/-GPU-/-NPU))
  - [6.1 CPU Inference](#6.1-CPU-Inference)
  - [6.2 GPU Inference](#6.2-GPU-Inference)
  - [6.3 NPU Inference](#6.3-NPU-Inference)
- [7. OpenVINO GenAI Integration](#7.-OpenVINO-GenAI-Integration)
- [8. Interactive Demo](#8.-Interactive-Demo)

## 1. Environment Setup
[back to top ⬆️](#Table-of-contents:)

Install all required dependencies: OpenVINO, OpenVINO GenAI, PyTorch, FunASR, and audio processing libraries.

In [20]:
# Fetch utility modules from openvino_notebooks repository
import requests
from pathlib import Path

utils = {
    # General OpenVINO notebook utilities
    "notebook_utils.py": "https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    "cmd_helper.py":     "https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py",
    "pip_helper.py":     "https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/pip_helper.py",
    # FunASR-specific helpers
    "ov_funasr_helper.py": "https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/funasr-nano/ov_funasr_helper.py",
    "gradio_helper.py":    "https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/funasr-nano/gradio_helper.py",
}

for filename, url in utils.items():
    if not Path(filename).exists():
        r = requests.get(url=url)
        r.raise_for_status()
        Path(filename).write_text(r.text)
        print(f"Downloaded {filename}")
    else:
        print(f"{filename} already exists")

notebook_utils.py already exists
cmd_helper.py already exists
pip_helper.py already exists
ov_funasr_helper.py already exists
gradio_helper.py already exists


In [ ]:
from cmd_helper import clone_repo
from pip_helper import pip_install
import platform

# Uninstall potentially conflicting packages before clean install
!pip uninstall -y -q torch torchaudio optimum-intel optimum

pip_install(
    "-q",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
    "torch",
    "nncf",
    "torchaudio",
    "openvino==2025.3.0",        # optimum-intel 1.26.x requires <2025.4
    "openvino-genai==2025.3.0.0",
    "optimum==2.1.0",
    "optimum-intel==1.26.1",     # last version compatible with openvino 2025.3
    "transformers>=4.51,<4.56",  # 4.51+ for Qwen3; <4.56 for optimum-intel 1.26
    "funasr>=1.2.7",
    "gradio",
    "huggingface_hub",
    "librosa",
)

# Clone the Fun-ASR repository (contains model.py needed for model loading)
repo_dir = Path("Fun-ASR")
revision = "efe63c122929bcca095fedc537c3081c5c4ee062"
clone_repo("https://github.com/FunAudioLLM/Fun-ASR.git", revision)

if platform.system() == "Darwin":
    pip_install("numpy<2.0")

## 2. Hugging Face Authentication
[back to top ⬆️](#Table-of-contents:)

FunASR Nano is a **public model** — no authentication token is required to download it. However, if you work with gated models in the future, you can authenticate using one of these methods:

- Set an environment variable: `export HF_TOKEN=your_token_here`
- Or run: `huggingface-cli login`

The cell below will use the token from the environment if available.

In [2]:
import os
from huggingface_hub import login

token = os.getenv("HF_TOKEN")
if token:
    login(token=token, add_to_git_credential=False)
    print("Logged into Hugging Face using HF_TOKEN")
else:
    print("No HF_TOKEN found — proceeding without authentication (OK for public models)")

No HF_TOKEN found — proceeding without authentication (OK for public models)


## 3. Model Download and Analysis
[back to top ⬆️](#Table-of-contents:)

FunASR Nano is available in two variants:

| Model | Languages | Training Data |
|---|---|---|
| **Fun-ASR-Nano-2512** | Chinese, English, Japanese + 7 dialects & 26 accents | Tens of millions of hours |
| **Fun-ASR-MLT-Nano-2512** | 31 languages (incl. East/Southeast Asian, European) | Hundreds of thousands of hours |

Both models share the same architecture (~800M parameters) based on **Qwen3-0.6B** as the language model backbone.

### Model architecture detail

The model processes audio through a multi-stage pipeline:

1. **WavFrontend**: Extracts 80-dimensional mel-filterbank features at 10ms frame shift
2. **Audio Encoder**: Convolutional + Transformer layers that downsample and encode audio features
3. **Audio Adaptor**: Projects encoder output to the LLM hidden dimension
4. **Embedding Merge**: Audio embeddings replace placeholder tokens in the text prompt, then are concatenated with text embeddings
5. **Qwen3-0.6B LLM**: Autoregressive decoder that generates the transcription

**Model inputs:**
- Raw audio waveform (WAV, MP3, etc.) at any sample rate (resampled internally to 16kHz)
- Text prompt template with `<|startofspeech|>...<|endofspeech|>` markers

**Model outputs:**
- Transcribed text string

In [3]:
import ipywidgets as widgets
from pathlib import Path

model_ids = ["FunAudioLLM/Fun-ASR-Nano-2512", "FunAudioLLM/Fun-ASR-MLT-Nano-2512"]

model_selector = widgets.Dropdown(
    options=model_ids,
    default=model_ids[0],
    description="Model:",
)

model_selector

Dropdown(description='Model:', options=('FunAudioLLM/Fun-ASR-Nano-2512', 'FunAudioLLM/Fun-ASR-MLT-Nano-2512'),…

In [4]:
from huggingface_hub import snapshot_download

model_name = model_selector.value.split("/")[-1]
model_dir = Path(model_name)
snapshot_download(repo_id=model_selector.value, local_dir=model_dir)

print(f"Model downloaded to: {model_dir}")
print(f"\nModel directory contents:")
for p in sorted(model_dir.iterdir()):
    if p.is_file():
        size_mb = p.stat().st_size / (1024 * 1024)
        print(f"  {p.name:40s} {size_mb:8.2f} MB")
    else:
        print(f"  {p.name}/")

Fetching 21 files:   0%|          | 0/21 [00:00<?, ?it/s]

Model downloaded to: Fun-ASR-Nano-2512

Model directory contents:
  .cache/
  .gitattributes                               0.00 MB
  Qwen3-0.6B/
  README.md                                    0.01 MB
  README_zh.md                                 0.01 MB
  config.yaml                                  0.00 MB
  configuration.json                           0.00 MB
  example/
  images/
  model.pt                                  1879.83 MB
  multilingual.tiktoken                        0.87 MB


In [5]:
# Inspect the PyTorch model structure
import sys, json
sys.path.insert(0, str(Path("Fun-ASR")))  # model.py lives in the cloned Fun-ASR repo

from model import FunASRNano

pt_model, kwargs = FunASRNano.from_pretrained(model=model_selector.value, device="cpu")
pt_model.eval()

print("=" * 60)
print("FunASR Nano — Model Architecture Analysis")
print("=" * 60)

# Frontend info
frontend = kwargs.get("frontend")
if frontend:
    print(f"\n[Audio Frontend]")
    print(f"  Type:         WavFrontend")
    print(f"  Sample rate:  {frontend.fs} Hz")
    print(f"  Frame shift:  {frontend.frame_shift} ms")
    print(f"  Frame length: {frontend.frame_length} ms")
    print(f"  Mel bins:     {frontend.n_mels}")
    print(f"  LFR M/N:      {frontend.lfr_m}/{frontend.lfr_n}")

# Audio encoder
print(f"\n[Audio Encoder]")
print(f"  Type: {pt_model.audio_encoder.__class__.__name__}")
total_enc_params = sum(p.numel() for p in pt_model.audio_encoder.parameters())
print(f"  Parameters: {total_enc_params / 1e6:.1f}M")

# Audio adaptor
if hasattr(pt_model, 'audio_adaptor'):
    print(f"\n[Audio Adaptor]")
    print(f"  Type: {pt_model.audio_adaptor.__class__.__name__}")
    total_adp_params = sum(p.numel() for p in pt_model.audio_adaptor.parameters())
    print(f"  Parameters: {total_adp_params / 1e6:.1f}M")

# LLM
print(f"\n[Language Model]")
print(f"  Type:          {pt_model.llm.__class__.__name__}")
print(f"  Config class:  {pt_model.llm.config.architectures}")
print(f"  Hidden size:   {pt_model.llm.config.hidden_size}")
print(f"  Layers:        {pt_model.llm.config.num_hidden_layers}")
print(f"  Attention heads: {pt_model.llm.config.num_attention_heads}")
print(f"  KV heads:      {pt_model.llm.config.num_key_value_heads}")
print(f"  Vocab size:    {pt_model.llm.config.vocab_size}")

total_params = sum(p.numel() for p in pt_model.parameters())
print(f"\n[Total model parameters: {total_params / 1e6:.1f}M]")

Loading remote code successfully: model
FunASR Nano — Model Architecture Analysis

[Audio Frontend]
  Type:         WavFrontend
  Sample rate:  16000 Hz
  Frame shift:  10 ms
  Frame length: 25 ms
  Mel bins:     80
  LFR M/N:      7/6

[Audio Encoder]
  Type: SenseVoiceEncoderSmall
  Parameters: 221.1M

[Audio Adaptor]
  Type: Transformer
  Parameters: 12.6M

[Language Model]
  Type:          Qwen3ForCausalLM
  Config class:  ['Qwen3ForCausalLM']
  Hidden size:   1024
  Layers:        28
  Attention heads: 16
  KV heads:      8
  Vocab size:    151936

[Total model parameters: 829.8M]


### Model I/O shapes analysis

Let's inspect the tensor shapes at each stage of the pipeline using a sample audio file.

In [6]:
import torch
from funasr.utils.load_utils import extract_fbank, load_audio_text_image_video

# Load a sample audio file
wav_path = str(model_dir / "example" / "en.mp3")
print(f"Sample audio: {wav_path}")

data_src = load_audio_text_image_video(wav_path, fs=frontend.fs)
speech, speech_lengths = extract_fbank(
    data_src, data_type="sound", frontend=frontend, is_final=True
)  # speech: [B, T, D]

print(f"\n--- Tensor shapes through the pipeline ---")
print(f"[Frontend output]")
print(f"  speech (Fbank features): {speech.shape}  (batch, time_frames, feat_dim)")
print(f"  speech_lengths:          {speech_lengths}")

# Run through encoder (expects [B, T, D])
with torch.no_grad():
    encoder_out, encoder_out_lens = pt_model.audio_encoder(speech, speech_lengths)

print(f"\n[Encoder output]")
print(f"  encoder_out:      {encoder_out.shape}  (batch, time_frames, hidden_dim)")
print(f"  encoder_out_lens: {encoder_out_lens}")

# Run through adaptor
if hasattr(pt_model, 'audio_adaptor'):
    with torch.no_grad():
        adapted_out, adapted_lens = pt_model.audio_adaptor(encoder_out, encoder_out_lens)
    print(f"\n[Adaptor output]")
    print(f"  adapted_out:      {adapted_out.shape}  (batch, time_frames, llm_hidden_dim)")
    print(f"  adapted_out_lens: {adapted_lens}")

# Token embeddings
tokenizer = kwargs["tokenizer"]
sample_text = "Hello world"
tokens = tokenizer.encode(sample_text, return_tensors="pt")
with torch.no_grad():
    text_embeds = pt_model.llm.model.get_input_embeddings()(tokens)
print(f"\n[Text embeddings]")
print(f"  input_ids:    {tokens.shape}")
print(f"  text_embeds:  {text_embeds.shape}  (batch, seq_len, llm_hidden_dim)")

print(f"\n[LLM output]")
print(f"  Generates tokens autoregressively -> decoded to text string")

Sample audio: Fun-ASR-Nano-2512/example/en.mp3

--- Tensor shapes through the pipeline ---
[Frontend output]
  speech (Fbank features): torch.Size([1, 120, 560])  (batch, time_frames, feat_dim)
  speech_lengths:          tensor([120], dtype=torch.int32)

[Encoder output]
  encoder_out:      torch.Size([1, 120, 512])  (batch, time_frames, hidden_dim)
  encoder_out_lens: tensor([120], dtype=torch.int32)

[Adaptor output]
  adapted_out:      torch.Size([1, 120, 1024])  (batch, time_frames, llm_hidden_dim)
  adapted_out_lens: tensor([120], dtype=torch.int32)

[Text embeddings]
  input_ids:    torch.Size([1, 2])
  text_embeds:  torch.Size([1, 2, 1024])  (batch, seq_len, llm_hidden_dim)

[LLM output]
  Generates tokens autoregressively -> decoded to text string


In [7]:
# Free PyTorch model memory before OpenVINO conversion
import gc

del pt_model
gc.collect()
print("PyTorch model freed from memory")

PyTorch model freed from memory


## 4. Conversion to OpenVINO IR
[back to top ⬆️](#Table-of-contents:)

FunASR Nano is a **multi-component model** that cannot be exported as a single OpenVINO IR. Instead, we convert each component separately:

| Component | File | Conversion method |
|---|---|---|
| Text Embeddings | `openvino_text_embeddings_model.xml` | `ov.convert_model` from PyTorch embedding layer |
| Audio Encoder + Adaptor | `openvino_encoder_model.xml` | `ov.convert_model` with wrapped forward |
| Language Model (Qwen3) | `openvino_model.xml` | `ov.convert_model` → make stateful (KV-cache) |

The conversion also saves the tokenizer and frontend configuration so the pipeline is self-contained.

The helper `convert_funasr()` from the OpenVINO Notebooks repository handles the full conversion pipeline.

In [21]:
# Apply Python 3.9 compatibility patch to ov_funasr_helper.py
# The helper uses X | Y union type syntax (PEP 604) which requires Python 3.10+.
# Adding `from __future__ import annotations` makes annotations lazily evaluated,
# fixing the runtime TypeError on Python 3.9.
helper_path = Path("ov_funasr_helper.py")
content = helper_path.read_text()
if not content.startswith("from __future__ import annotations"):
    helper_path.write_text("from __future__ import annotations\n\n" + content)
    print("Applied Python 3.9 compatibility patch to ov_funasr_helper.py")
else:
    print("ov_funasr_helper.py already patched")

ov_funasr_helper.py already patched


In [9]:
from ov_funasr_helper import convert_funasr

ov_model_dir = Path(model_name + "-ov")
convert_funasr(str(model_dir), ov_model_dir)

# Show resulting IR files
print(f"\nOpenVINO IR files in {ov_model_dir}:")
for p in sorted(ov_model_dir.glob("*.xml")):
    bin_path = p.with_suffix(".bin")
    bin_size = bin_path.stat().st_size / (1024 * 1024) if bin_path.exists() else 0
    print(f"  {p.name:45s} weights: {bin_size:.1f} MB")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


[OK] Fun-ASR-Nano-2512 model already converted. You can find results in Fun-ASR-Nano-2512-ov

OpenVINO IR files in Fun-ASR-Nano-2512-ov:
  openvino_encoder_model.xml                    weights: 445.8 MB
  openvino_model.xml                            weights: 1136.9 MB
  openvino_text_embeddings_model.xml            weights: 296.8 MB


/home/pkrzemin/tasks/benchmark/venv/lib/python3.9/site-packages/openvino/runtime/__init__.py:10: DeprecationWarning: The `openvino.runtime` module is deprecated and will be removed in the 2026.0 release. Please replace `openvino.runtime` with `openvino`.
  warnings.warn(


### Inspect OpenVINO IR models

Let's verify the converted IR files and examine their input/output specifications.

In [10]:
import openvino as ov

core = ov.Core()

ir_files = {
    "Text Embeddings": ov_model_dir / "openvino_text_embeddings_model.xml",
    "Audio Encoder":   ov_model_dir / "openvino_encoder_model.xml",
    "Language Model":  ov_model_dir / "openvino_model.xml",
}

def tensor_name(t):
    try:
        return t.get_any_name()
    except RuntimeError:
        return "(unnamed)"

for name, xml_path in ir_files.items():
    model = core.read_model(xml_path)
    print(f"\n{'=' * 50}")
    print(f"[{name}] — {xml_path.name}")
    print(f"{'=' * 50}")
    print(f"  Inputs ({len(model.inputs)}):")
    for inp in model.inputs:
        print(f"    {tensor_name(inp):35s} shape={inp.get_partial_shape()}  dtype={inp.get_element_type()}")
    print(f"  Outputs ({len(model.outputs)}):")
    for out in model.outputs[:5]:  # show first 5 to avoid flooding
        print(f"    {tensor_name(out):35s} shape={out.get_partial_shape()}  dtype={out.get_element_type()}")
    if len(model.outputs) > 5:
        print(f"    ... and {len(model.outputs) - 5} more outputs (KV-cache)")
    if len(model.get_sinks()) > 0:
        print(f"  Stateful: Yes ({len(model.get_sinks())} state variables — KV-cache hidden inside model)")
    del model


[Text Embeddings] — openvino_text_embeddings_model.xml
  Inputs (1):
    input                               shape=[?,?]  dtype=<Type: 'int32_t'>
  Outputs (1):
    (unnamed)                           shape=[?,?,1024]  dtype=<Type: 'float32'>

[Audio Encoder] — openvino_encoder_model.xml
  Inputs (2):
    speech                              shape=[?,?,?]  dtype=<Type: 'float32'>
    speech_lengths                      shape=[?]  dtype=<Type: 'int32_t'>
  Outputs (2):
    (unnamed)                           shape=[?,?,1024]  dtype=<Type: 'float32'>
    lengths.1                           shape=[?]  dtype=<Type: 'int32_t'>

[Language Model] — openvino_model.xml
  Inputs (4):
    attention_mask                      shape=[?,?]  dtype=<Type: 'int64_t'>
    position_ids                        shape=[?,?]  dtype=<Type: 'int64_t'>
    inputs_embeds                       shape=[?,?,1024]  dtype=<Type: 'float32'>
    beam_idx                            shape=[?]  dtype=<Type: 'int32_t'>
  Outp

## 5. OpenVINO Runtime Inference
[back to top ⬆️](#Table-of-contents:)

Now we load the converted models and run inference using the `OVFunASRNano` wrapper class. This class:
- Loads all three IR components (text embeddings, encoder, LLM)
- Orchestrates the full pipeline: audio preprocessing -> encoding -> embedding merge -> LLM generation
- Uses `OVModelForCausalLMWithEmbed` to support `inputs_embeds` input for the LLM (needed for multimodal fusion)

### Select Inference Device

In [11]:
from notebook_utils import device_widget

device = device_widget("CPU", exclude=["AUTO"])

device

Dropdown(description='Device:', options=('CPU',), value='CPU')

In [12]:
# Device-specific LLM configuration
llm_ov_config = {
    "CPU": {},
    "GPU": {"ACTIVATIONS_SCALE_FACTOR": "8.0"},
    "NPU": {
        "ACTIVATIONS_SCALE_FACTOR": "8.0",
        "NPU_USE_NPUW": "YES",
        "NPUW_LLM": "YES",
        "NPUW_ONLINE_PIPELINE": "NONE",
        "MAX_PROMPT_LEN": 1024,
        "NPUW_LLM_MIN_RESPONSE_LEN": 512,
    },
}

In [13]:
from ov_funasr_helper import OVFunASRNano

ov_model = OVFunASRNano(ov_model_dir, device=device.value, llm_ov_config=llm_ov_config[device.value])

[OK] Tokenizer loaded from Fun-ASR-Nano-2512-ov
[OK] Frontend and inference config loaded from Fun-ASR-Nano-2512-ov/frontend_config.json


### Run Speech Recognition

Let's transcribe a sample English audio file included with the model.

In [14]:
import time

# Transcribe English sample
wav_path_en = str(model_dir / "example" / "en.mp3")
print(f"Audio file: {wav_path_en}\n")

start = time.perf_counter()
res, meta_data = ov_model.inference(data_in=[wav_path_en])
elapsed = time.perf_counter() - start

text = res[0]["text"]
print(f"Transcription: {text}")
print(f"\nInference time: {elapsed:.2f}s")
if "batch_data_time" in meta_data:
    print(f"Audio duration:  {meta_data['batch_data_time']:.2f}s")
    print(f"Real-time factor: {elapsed / meta_data['batch_data_time']:.2f}x")

Audio file: Fun-ASR-Nano-2512/example/en.mp3



The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Transcription: The tribal chieftain called for the boy, and presented him with fifty pieces of gold.

Inference time: 1.50s
Audio duration:  7.20s
Real-time factor: 0.21x


In [15]:
# Transcribe Chinese sample (if available)
wav_path_zh = str(model_dir / "example" / "zh.mp3")
if Path(wav_path_zh).exists():
    start = time.perf_counter()
    res_zh, meta_zh = ov_model.inference(data_in=[wav_path_zh])
    elapsed = time.perf_counter() - start
    print(f"Chinese transcription: {res_zh[0]['text']}")
    print(f"Inference time: {elapsed:.2f}s")
else:
    print("Chinese sample not available in this model variant")

Chinese transcription: 开饭时间早上九点至下午五点。
Inference time: 0.82s


## 6. Multi-Device Inference (CPU / GPU / NPU)
[back to top ⬆️](#Table-of-contents:)

OpenVINO supports multiple hardware backends. The following subsections validate inference on CPU and provide ready-to-run code for GPU and NPU devices.

In [16]:
import openvino as ov
import time

core = ov.Core()
available_devices = core.available_devices
print(f"Available OpenVINO devices: {available_devices}")

wav_test = str(model_dir / "example" / "en.mp3")

# --- CPU inference (always available) ----------------------------------------
print("\n" + "=" * 55)
print("6.1  CPU Inference")
print("=" * 55)

ov_cpu = OVFunASRNano(ov_model_dir, device="CPU", llm_ov_config={})
start = time.perf_counter()
res_cpu, _ = ov_cpu.inference(data_in=[wav_test])
cpu_time = time.perf_counter() - start
print(f"  Result: {res_cpu[0]['text']}")
print(f"  Time:   {cpu_time:.2f}s")
del ov_cpu

Available OpenVINO devices: ['CPU']

6.1  CPU Inference
[OK] Tokenizer loaded from Fun-ASR-Nano-2512-ov
[OK] Frontend and inference config loaded from Fun-ASR-Nano-2512-ov/frontend_config.json
  Result: The tribal chieftain called for the boy and presented him with fifty pieces of gold.
  Time:   1.58s


### 6.2  GPU Inference

Intel Xe / Arc / Iris Xe GPUs are supported via the `GPU` OpenVINO plugin.
Run the cell below on a machine with an Intel GPU to compare throughput against CPU.

Key configuration knob:
- `ACTIVATIONS_SCALE_FACTOR` (`"8.0"`) - scales activations to reduce numeric range overflow on GPU, improving accuracy for quantized attention layers.

> **Tip**: The first run may be slower due to kernel compilation. Use `CACHE_DIR` to persist compiled kernels across sessions.

In [17]:
gpu_llm_config = {
    "ACTIVATIONS_SCALE_FACTOR": "8.0",
    "CACHE_DIR": ".ovms_cache_gpu",
}

if "GPU" in core.available_devices:
    print("=" * 55)
    print("GPU Inference")
    print("=" * 55)
    try:
        ov_gpu = OVFunASRNano(
            model_dir=ov_model_dir,
            device="GPU",
            llm_ov_config=gpu_llm_config,
        )
        wav_test_en = str(model_dir / "example" / "en.mp3")
        wav_test_zh = str(model_dir / "example" / "zh.mp3")

        start = time.perf_counter()
        res_gpu_en, _ = ov_gpu.inference(data_in=[wav_test_en])
        gpu_time_en = time.perf_counter() - start
        print(f"  [EN] Result: {res_gpu_en[0]['text']}")
        print(f"       Time:   {gpu_time_en:.2f}s")

        start = time.perf_counter()
        res_gpu_zh, _ = ov_gpu.inference(data_in=[wav_test_zh])
        gpu_time_zh = time.perf_counter() - start
        print(f"  [ZH] Result: {res_gpu_zh[0]['text']}")
        print(f"       Time:   {gpu_time_zh:.2f}s")

        del ov_gpu
    except Exception as e:
        print(f"GPU inference failed: {e}")
else:
    print(f"No GPU device found. Available devices: {core.available_devices}")
    print("To run GPU inference, use a machine with an Intel GPU (Arc, Iris Xe, UHD).")

No GPU device found. Available devices: ['CPU']
To run GPU inference, use a machine with an Intel GPU (Arc, Iris Xe, UHD).


### 6.3  NPU Inference

Intel NPU (Neural Processing Unit) is available on Intel Core Ultra (Series 1 / 2), Meteor Lake, and Lunar Lake processors.

For FunASR Nano, **the LLM component runs on the NPU** while the audio encoder runs on CPU (NPU does not support dynamic shapes required by the encoder). This hybrid approach can reduce CPU load significantly.

> **Note**: NPU compilation can take 30-60 seconds on the first run. Use `CACHE_DIR` to cache compiled models for instant re-use.

In [18]:
npu_llm_config = {
    "CACHE_DIR": ".ovms_cache_npu",
    # Uncomment the line below for higher numerical precision on NPU layers:
    # "NPU_COMPILATION_MODE_PARAMS": "compute-layers-with-higher-precision=Sqrt,Power,ReduceMean,Add",
}

if "NPU" in core.available_devices:
    print("=" * 55)
    print("NPU Inference  (LLM → NPU | encoder → CPU)")
    print("=" * 55)
    try:
        ov_npu = OVFunASRNano(
            model_dir=ov_model_dir,
            device="NPU",
            llm_ov_config=npu_llm_config,
        )
        wav_test_en = str(model_dir / "example" / "en.mp3")
        wav_test_zh = str(model_dir / "example" / "zh.mp3")

        start = time.perf_counter()
        res_npu_en, _ = ov_npu.inference(data_in=[wav_test_en])
        npu_time_en = time.perf_counter() - start
        print(f"  [EN] Result: {res_npu_en[0]['text']}")
        print(f"       Time:   {npu_time_en:.2f}s")

        start = time.perf_counter()
        res_npu_zh, _ = ov_npu.inference(data_in=[wav_test_zh])
        npu_time_zh = time.perf_counter() - start
        print(f"  [ZH] Result: {res_npu_zh[0]['text']}")
        print(f"       Time:   {npu_time_zh:.2f}s")

        del ov_npu
    except Exception as e:
        print(f"NPU inference failed: {e}")
else:
    print(f"No NPU device found. Available devices: {core.available_devices}")
    print("To run NPU inference, use an Intel Core Ultra (Meteor Lake / Lunar Lake) CPU.")

No NPU device found. Available devices: ['CPU']
To run NPU inference, use an Intel Core Ultra (Meteor Lake / Lunar Lake) CPU.


## 7. OpenVINO GenAI Integration
[back to top ⬆️](#Table-of-contents:)

### Can FunASR Nano use OpenVINO GenAI?

OpenVINO GenAI provides high-level pipeline APIs for common model types. The relevant pipelines for an ASR model are:

| GenAI Pipeline | Architecture | Applicable to FunASR? |
|---|---|---|
| `WhisperPipeline` | Whisper encoder-decoder with cross-attention | No - Different architecture |
| `LLMPipeline` | Text-only autoregressive LLMs (`input_ids`) | No - Needs `inputs_embeds` |
| `VLMPipeline` | Vision-Language models (image + text) | No - Not audio-based |

The following cells **actually attempt** each pipeline and capture the resulting errors to show exactly why they fail.

---

**Why `WhisperPipeline` fails:**

Whisper is a self-contained encoder-decoder where the encoder processes mel spectrograms and the decoder attends to encoder outputs via cross-attention. `WhisperPipeline` expects this specific two-model structure (e.g. `encoder_model.xml` + `decoder_model.xml`).

FunASR is fundamentally different: its audio encoder produces embeddings that are **spliced directly into the LLM's token embedding sequence** before being fed to a standard causal LM. There is no cross-attention - the architecture is closer to a Vision-Language Model than to Whisper.

**Why `LLMPipeline` fails:**

The Qwen3-0.6B backbone *is* a standard causal LM, so at first glance `LLMPipeline` looks promising. However `LLMPipeline.generate()` only accepts raw text or `input_ids` - it has **no `inputs_embeds` pathway**. FunASR requires passing audio embeddings as `inputs_embeds` so they can be merged with text token embeddings before the first transformer layer. Without that, audio context cannot reach the LLM.

In [19]:
import openvino_genai

print("OpenVINO GenAI version:", openvino_genai.__version__)
print()
print("Available GenAI pipeline classes:")
for attr in sorted(dir(openvino_genai)):
    if "Pipeline" in attr:
        print(f"  - {attr}")

# -----------------------------------------------------------------------------
# Attempt 1: WhisperPipeline on the FunASR IR directory
# -----------------------------------------------------------------------------
print()
print("=" * 70)
print("Attempt 1: openvino_genai.WhisperPipeline(ov_model_dir, 'CPU')")
print("=" * 70)
print("Expected: FAILS -- WhisperPipeline needs encoder_model.xml / decoder_model.xml")
print("          FunASR uses openvino_encoder_model.xml + openvino_model.xml")
print()
try:
    whisper_pipe = openvino_genai.WhisperPipeline(str(ov_model_dir), "CPU")
    import numpy as np
    import soundfile as sf
    audio, sr = sf.read(wav_path_en, dtype="float32")
    result = whisper_pipe.generate(audio.tolist())
    print(f"  [Unexpected success] Result: {result}")
except Exception as e:
    print(f"  [FAIL] {type(e).__name__}: {e}")

# -----------------------------------------------------------------------------
# Attempt 2: LLMPipeline on the FunASR IR directory
# -----------------------------------------------------------------------------
print()
print("=" * 70)
print("Attempt 2: openvino_genai.LLMPipeline(ov_model_dir, 'CPU')")
print("=" * 70)
print("Expected: FAILS or gives wrong answers -- openvino_model.xml uses")
print("          'inputs_embeds' not 'input_ids'; audio context is not injected.")
print()
try:
    llm_pipe = openvino_genai.LLMPipeline(str(ov_model_dir), "CPU")
    print("  Pipeline loaded. Attempting text-only generation (no audio)...")
    text_out = llm_pipe.generate(
        "Transcribe the following audio:",
        openvino_genai.GenerationConfig(max_new_tokens=20),
    )
    print(f"  Output (no audio context): {text_out!r}")
    print()
    print("  NOTE: Even if loading succeeds, there is no way to pass audio into")
    print("        LLMPipeline -- it only accepts text/input_ids, not inputs_embeds.")
except Exception as e:
    print(f"  [FAIL] {type(e).__name__}: {e}")

print()
print("=" * 70)
print("Result: FunASR Nano cannot be used with any standard GenAI pipeline.")
print("        The OVFunASRNano wrapper in Section 5 is the correct approach.")
print("=" * 70)

OpenVINO GenAI version: 2025.3.0.0-2463-3c0e2d3e7e1

Available GenAI pipeline classes:
  - ContinuousBatchingPipeline
  - Image2ImagePipeline
  - InpaintingPipeline
  - LLMPipeline
  - Text2ImagePipeline
  - Text2SpeechPipeline
  - TextEmbeddingPipeline
  - TextRerankPipeline
  - VLMPipeline
  - WhisperPipeline

Attempt 1: openvino_genai.WhisperPipeline(ov_model_dir, 'CPU')
Expected: FAILS -- WhisperPipeline needs encoder_model.xml / decoder_model.xml
          FunASR uses openvino_encoder_model.xml + openvino_model.xml

  [FAIL] RuntimeError: Exception from src/inference/src/cpp/core.cpp:126:
Exception from src/inference/src/dev/plugin.cpp:58:
Check 'consumer.get_expr()->get_loop_ids() == loop_ids' failed at src/common/snippets/src/lowered/pass/move_scalar_to_consumer.cpp:34:
All consumers of a Scalar expression are expected to have the same loop IDs




Attempt 2: openvino_genai.LLMPipeline(ov_model_dir, 'CPU')
Expected: FAILS or gives wrong answers -- openvino_model.xml uses
       

## 8. Interactive Demo
[back to top ⬆️](#Table-of-contents:)

Launch a Gradio interface for interactive audio transcription. You can upload audio files or record from microphone.

In [ ]:
from gradio_helper import make_demo

demo = make_demo(ov_model, model_dir)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(debug=True, share=True)